# Step 2 — Government Quality Score (MIPS)

**Business question:** Given a list of Massachusetts provider NPIs, what is each provider's CMS MIPS quality score?

**Logic:** Pull QPP/MIPS final scores, enrich with Care Compare and Part B utilization data, and produce a clean per-NPI table. Business logic (three-situation classification) will be designed after exploring this data.

**Structural ceiling:** All CMS data is Medicare fee-for-service. Providers with mostly commercial patients will look thin in this data. This is a known limitation, not something we can engineer around.

## Step 1 — Load Data

**What this does:** Load the four datasets needed for the MIPS quality score pipeline:
1. **QPP/MIPS Final Scores** — CMS's official quality assessment per provider
2. **Care Compare** — clinician-level enrichment (telehealth, affiliations)
3. **Part B Utilization** — billing volume and specialty data
4. **NPPES** — provider registry filtered to Massachusetts (our master NPI list)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import glob

# --- File detection ---
# QPP/MIPS: try known filename patterns
qpp_candidates = glob.glob("2_datasets/ec_score*.csv") + glob.glob("2_datasets/MIPS*2023*.csv") + glob.glob("2_datasets/*qpp*.csv") + glob.glob("2_datasets/*QPP*.csv")
if not qpp_candidates:
    raise FileNotFoundError(
        "QPP/MIPS file not found. Expected 'ec_score_file.csv' or similar in 2_datasets/. "
        "Download from https://data.cms.gov/provider-data/topics/doctors-clinicians (QPP section)"
    )
qpp_file = qpp_candidates[0]
print(f"Using QPP file: {qpp_file}")

In [ ]:
# Load QPP/MIPS final scores
qpp = pd.read_csv(qpp_file, dtype=str)
print(f"QPP/MIPS: {qpp.shape[0]:,} rows, {qpp.shape[1]} columns")
print(f"Columns: {list(qpp.columns)}")

# Sanity check: expect ~800k-1M rows nationally
if qpp.shape[0] < 500_000:
    print(f"⚠️ WARNING: Only {qpp.shape[0]:,} rows. Expected ~800k-1M. Check if this is the right file/year.")
elif qpp.shape[0] > 2_000_000:
    print(f"⚠️ WARNING: {qpp.shape[0]:,} rows is unusually high. May contain multiple years.")
else:
    print(f"✓ Row count looks reasonable for national MIPS data")

In [ ]:
# Load Care Compare clinician data
cc_candidates = glob.glob("2_datasets/DAC_National*.csv") + glob.glob("2_datasets/*care_compare*.csv")
if not cc_candidates:
    raise FileNotFoundError(
        "Care Compare file not found. Expected 'DAC_NationalDownloadableFile.csv' in 2_datasets/. "
        "Download from https://data.cms.gov/provider-data/topics/doctors-clinicians"
    )
cc_file = cc_candidates[0]
print(f"Using Care Compare file: {cc_file}")

care_compare = pd.read_csv(cc_file, dtype=str)
print(f"Care Compare: {care_compare.shape[0]:,} rows, {care_compare.shape[1]} columns")
print(f"Columns: {list(care_compare.columns)}")

In [ ]:
# Load Medicare Part B utilization
partb_candidates = (
    glob.glob("2_datasets/MUP_PHY*Prov_Svc*.csv")
    + glob.glob("2_datasets/Medicare_Physician*Provider_and_Service*.csv")
)
if not partb_candidates:
    raise FileNotFoundError(
        "Part B file not found. Expected 'MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv' in 2_datasets/. "
        "Download from https://data.cms.gov/provider-summary-by-type-of-service/"
    )
partb_file = partb_candidates[0]
print(f"Using Part B file: {partb_file}")
print(f"File size: {os.path.getsize(partb_file) / 1e9:.1f} GB")

# Load with only needed columns to save memory (~3GB file)
partb_usecols = [
    "Rndrng_NPI", "Rndrng_Prvdr_Last_Org_Name", "Rndrng_Prvdr_First_Name",
    "Rndrng_Prvdr_Type", "Rndrng_Prvdr_State_Abrvtn",
    "HCPCS_Cd", "HCPCS_Desc", "Place_Of_Srvc",
    "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
    "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt"
]
partb = pd.read_csv(partb_file, dtype=str, usecols=partb_usecols)
print(f"Part B: {partb.shape[0]:,} rows, {partb.shape[1]} columns")

# Sanity check
if partb.shape[0] < 5_000_000:
    print(f"⚠️ WARNING: Only {partb.shape[0]:,} rows. National Part B by-service typically has 9M+ rows.")
else:
    print(f"✓ Row count looks reasonable for national Part B data")

In [ ]:
# Load NPPES — chunked, filter to MA on the fly
nppes_candidates = glob.glob("2_datasets/NPPES Data/npidata_pfile_*.csv")
# Exclude the fileheader file
nppes_candidates = [f for f in nppes_candidates if "fileheader" not in f]
if not nppes_candidates:
    raise FileNotFoundError(
        "NPPES file not found. Expected 'npidata_pfile_*.csv' in 2_datasets/NPPES Data/. "
        "Download from https://download.cms.gov/nppes/NPI_Files.html and extract the ZIP."
    )
nppes_file = nppes_candidates[0]
print(f"Using NPPES file: {nppes_file}")
print(f"File size: {os.path.getsize(nppes_file) / 1e9:.1f} GB")

# Key columns to keep (minimizes memory — full file has 330 columns)
nppes_cols = [
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Credential Text",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address Postal Code",
    "Healthcare Provider Taxonomy Code_1",
    "Provider Enumeration Date",
    "NPI Deactivation Date",
]

# Chunked load, filter to MA
print("Loading NPPES and filtering to MA (this takes a few minutes)...")
ma_chunks = []
total_rows = 0
for chunk in pd.read_csv(nppes_file, dtype=str, usecols=nppes_cols, chunksize=50_000):
    total_rows += len(chunk)
    ma_chunk = chunk[
        chunk["Provider Business Practice Location Address State Name"]
        .str.upper().str.strip()
        .isin(["MA", "MASSACHUSETTS"])
    ]
    if len(ma_chunk) > 0:
        ma_chunks.append(ma_chunk)

nppes_ma = pd.concat(ma_chunks, ignore_index=True)
print(f"NPPES total rows processed: {total_rows:,}")
print(f"NPPES MA providers: {nppes_ma.shape[0]:,}")

# Sanity check: expect ~80k-120k MA NPIs
if nppes_ma.shape[0] < 30_000:
    print(f"⚠️ WARNING: Only {nppes_ma.shape[0]:,} MA NPIs. Expected ~80k-120k. Check filter column.")
elif nppes_ma.shape[0] > 200_000:
    print(f"⚠️ WARNING: {nppes_ma.shape[0]:,} MA NPIs is unusually high. Filter may not be working.")
else:
    print(f"✓ MA NPI count looks reasonable")

In [ ]:
# Load Medicare Part D Prescriber data (by Provider, 2023)
partd_candidates = glob.glob("2_datasets/MUP_DPR*DY23*NPI*.csv")
if not partd_candidates:
    raise FileNotFoundError(
        "Part D Prescriber file not found. Expected 'MUP_DPR_RY25_P04_V10_DY23_NPI.csv' in 2_datasets/. "
        "Download from https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers"
    )
partd_file = partd_candidates[0]
print(f"Using Part D file: {partd_file}")

# Load with key columns only — include Prscrbr_Ent_Cd to filter out organizations
partd_usecols = [
    "PRSCRBR_NPI", "Prscrbr_Last_Org_Name", "Prscrbr_First_Name",
    "Prscrbr_Ent_Cd", "Prscrbr_Type", "Prscrbr_State_Abrvtn",
    "Tot_Clms", "Tot_30day_Fills", "Tot_Drug_Cst", "Tot_Day_Suply", "Tot_Benes",
    "Brnd_Tot_Clms", "Brnd_Tot_Drug_Cst", "Gnrc_Tot_Clms", "Gnrc_Tot_Drug_Cst",
    "Opioid_Tot_Clms", "Opioid_Tot_Drug_Cst", "Opioid_Prscrbr_Rate",
    "Opioid_LA_Tot_Clms", "Opioid_LA_Prscrbr_Rate",
    "Antbtc_Tot_Clms", "Antbtc_Tot_Drug_Cst",
    "Bene_Avg_Age", "Bene_Avg_Risk_Scre",
]
partd = pd.read_csv(partd_file, dtype=str, usecols=partd_usecols)
print(f"Part D: {partd.shape[0]:,} rows, {partd.shape[1]} columns")

# Entity type distribution (I=Individual, O=Organization)
print(f"\nEntity type distribution:")
print(partd["Prscrbr_Ent_Cd"].value_counts().to_string())

if partd.shape[0] < 800_000:
    print(f"\nWARNING: Only {partd.shape[0]:,} rows. Expected ~1.1M+.")
else:
    print(f"\nRow count looks reasonable")

In [ ]:
# Load CMS Facility Affiliations (NPI -> Hospital CCN crosswalk)
facility_affil = pd.read_csv("2_datasets/Facility_Affiliation.csv", dtype=str,
                              usecols=["NPI", "facility_type", "Facility Affiliations Certification Number"])
print(f"Facility Affiliations: {facility_affil.shape[0]:,} rows")
print(f"\nFacility type distribution:")
print(facility_affil["facility_type"].value_counts().to_string())
print(f"\nUnique NPIs: {facility_affil['NPI'].nunique():,}")
print(f"Unique CCNs: {facility_affil['Facility Affiliations Certification Number'].nunique():,}")

In [ ]:
# Load Hospital Quality files and build per-facility summary
# All keyed by Facility ID (6-digit CCN)

hosp_general = pd.read_csv("2_datasets/Hospital_General_Information.csv", dtype=str,
    usecols=["Facility ID", "Facility Name", "State", "Hospital overall rating", "Hospital Type", "Hospital Ownership"])
print(f"Hospital General Info: {hosp_general.shape[0]:,} facilities")

hcahps = pd.read_csv("2_datasets/HCAHPS-Hospital.csv", dtype=str,
    usecols=["Facility ID", "HCAHPS Measure ID", "Patient Survey Star Rating"])
print(f"HCAHPS: {hcahps.shape[0]:,} rows")

infections = pd.read_csv("2_datasets/Healthcare_Associated_Infections-Hospital.csv", dtype=str,
    usecols=["Facility ID", "Compared to National"])
print(f"Infections: {infections.shape[0]:,} rows")

complications = pd.read_csv("2_datasets/Complications_and_Deaths-Hospital.csv", dtype=str,
    usecols=["Facility ID", "Compared to National"])
print(f"Complications: {complications.shape[0]:,} rows")

readmissions = pd.read_csv("2_datasets/Unplanned_Hospital_Visits-Hospital.csv", dtype=str,
    usecols=["Facility ID", "Compared to National"])
print(f"Readmissions: {readmissions.shape[0]:,} rows")

hvbp = pd.read_csv("2_datasets/hvbp_tps.csv", dtype=str,
    usecols=["Facility ID", "Total Performance Score"])
print(f"HVBP TPS: {hvbp.shape[0]:,} rows")

# Build single hospital quality summary: one row per Facility ID
hosp_summary = hosp_general[["Facility ID", "Facility Name", "State", "Hospital overall rating",
                              "Hospital Type", "Hospital Ownership"]].copy()
hosp_summary = hosp_summary.rename(columns={"Hospital overall rating": "hospital_star_rating"})

# HVBP Total Performance Score
hvbp_scores = hvbp[["Facility ID", "Total Performance Score"]].rename(
    columns={"Total Performance Score": "hvbp_total_score"})
hosp_summary = hosp_summary.merge(hvbp_scores, on="Facility ID", how="left")

# HCAHPS: overall hospital rating star
hcahps_overall = hcahps[hcahps["HCAHPS Measure ID"] == "H_STAR_RATING"][
    ["Facility ID", "Patient Survey Star Rating"]].rename(
    columns={"Patient Survey Star Rating": "hcahps_star_rating"})
hosp_summary = hosp_summary.merge(hcahps_overall, on="Facility ID", how="left")

# Infections: count of "Worse" measures (exact string from data)
inf_worse = infections[infections["Compared to National"] == "Worse than the National Benchmark"]
inf_count = inf_worse.groupby("Facility ID").size().reset_index(name="infection_measures_worse")
hosp_summary = hosp_summary.merge(inf_count, on="Facility ID", how="left")
hosp_summary["infection_measures_worse"] = hosp_summary["infection_measures_worse"].fillna(0).astype(int)

# Complications: count of "Worse" measures (two string variants in data)
comp_worse = complications[complications["Compared to National"].isin([
    "Worse Than the National Rate", "Worse Than the National Value"])]
comp_count = comp_worse.groupby("Facility ID").size().reset_index(name="complication_measures_worse")
hosp_summary = hosp_summary.merge(comp_count, on="Facility ID", how="left")
hosp_summary["complication_measures_worse"] = hosp_summary["complication_measures_worse"].fillna(0).astype(int)

# Readmissions: count of "Worse" measures (two string variants in data)
readm_worse = readmissions[readmissions["Compared to National"].isin([
    "Worse Than the National Rate", "Worse than expected"])]
readm_count = readm_worse.groupby("Facility ID").size().reset_index(name="readmission_measures_worse")
hosp_summary = hosp_summary.merge(readm_count, on="Facility ID", how="left")
hosp_summary["readmission_measures_worse"] = hosp_summary["readmission_measures_worse"].fillna(0).astype(int)

print(f"\nHospital summary: {len(hosp_summary):,} facilities, {hosp_summary.shape[1]} columns")
print(f"Star rating distribution:")
print(hosp_summary["hospital_star_rating"].value_counts().sort_index().to_string())

In [ ]:
# Load Open Payments (General Payments, 2023) — ~8GB, chunked aggregation
op_file = "2_datasets/OP_DTL_GNRL_PGYR2023_P01302025_01212025.csv"
if not os.path.exists(op_file):
    raise FileNotFoundError(f"Open Payments file not found at {op_file}")
print(f"Using Open Payments file: {op_file}")
print(f"File size: {os.path.getsize(op_file) / 1e9:.1f} GB")

# Chunked aggregation: aggregate each chunk with pandas groupby, then concat and re-aggregate
op_usecols = [
    "Covered_Recipient_NPI",
    "Total_Amount_of_Payment_USDollars",
    "Nature_of_Payment_or_Transfer_of_Value",
]

print("Aggregating Open Payments per NPI (this takes a few minutes)...")
op_chunks = []
total_rows = 0
for chunk in pd.read_csv(op_file, dtype=str, usecols=op_usecols, chunksize=100_000):
    total_rows += len(chunk)
    chunk["_amount"] = pd.to_numeric(chunk["Total_Amount_of_Payment_USDollars"], errors="coerce")
    chunk["_is_food_bev"] = (chunk["Nature_of_Payment_or_Transfer_of_Value"] == "Food and Beverage").astype(int)
    chunk["_is_consulting"] = chunk["Nature_of_Payment_or_Transfer_of_Value"].str.contains(
        "Consulting", case=False, na=False).astype(int)

    agg = chunk.groupby("Covered_Recipient_NPI").agg(
        total_payment_amount=("_amount", "sum"),
        payment_count=("_amount", "count"),
        food_bev_count=("_is_food_bev", "sum"),
        consulting_count=("_is_consulting", "sum"),
    )
    op_chunks.append(agg)

# Concat and re-aggregate across chunks
open_payments = pd.concat(op_chunks).groupby(level=0).sum().reset_index()
open_payments = open_payments.rename(columns={"Covered_Recipient_NPI": "NPI"})

print(f"Open Payments rows processed: {total_rows:,}")
print(f"Unique NPIs with payments: {len(open_payments):,}")
print(f"\nPayment stats:")
print(open_payments["total_payment_amount"].describe().to_string())

## Step 2 — Exploratory Data Analysis

**What this does:** Before cleaning or joining anything, understand what we're working with. Schema, shape, null rates, distributions, and key charts for each dataset.

In [ ]:
# --- QPP/MIPS EDA ---
print("=" * 60)
print("QPP/MIPS FINAL SCORES")
print("=" * 60)

print(f"\nShape: {qpp.shape}")
print(f"\nColumns: {list(qpp.columns)}")

# Note: most QPP columns have a leading space in the name
# Strip column names for easier access
qpp.columns = qpp.columns.str.strip()
print(f"\nColumns (cleaned): {list(qpp.columns)}")

print(f"\nNull rates:")
print((qpp.isnull().sum() / len(qpp) * 100).round(1).to_string())

print(f"\nData types:")
print(qpp.dtypes.to_string())

print(f"\nUnique NPIs: {qpp['NPI'].nunique():,}")
print(f"Source distribution:")
print(qpp["source"].value_counts().to_string())

In [ ]:
# QPP visualizations

# MIPS final score distribution
qpp_scores = pd.to_numeric(qpp["final_MIPS_score"], errors="coerce")
fig1 = px.histogram(
    qpp_scores.dropna(),
    nbins=50,
    title="QPP: MIPS Final Score Distribution (National)",
    labels={"value": "MIPS Final Score", "count": "Providers"},
)
fig1.show()

# Category score distributions
category_cols = {
    "Quality_category_score": "Quality",
    "PI_category_score": "Promoting Interoperability",
    "IA_category_score": "Improvement Activities",
    "Cost_category_score": "Cost",
}
for col, label in category_cols.items():
    vals = pd.to_numeric(qpp[col], errors="coerce").dropna()
    if len(vals) > 0:
        fig = px.histogram(vals, nbins=30, title=f"QPP: {label} Category Score Distribution")
        fig.show()
    print(f"{label}: {len(vals):,} non-null values, mean={vals.mean():.1f}, median={vals.median():.1f}")

# No low-volume flag column — will infer from Part B
print("\nNo explicit low-volume flag in QPP data. Will infer from Part B billing volume.")

In [ ]:
# --- Care Compare EDA ---
print("=" * 60)
print("CARE COMPARE — CLINICIAN DATA")
print("=" * 60)

print(f"\nShape: {care_compare.shape}")
print(f"\nColumns: {list(care_compare.columns)}")

print(f"\nNull rates:")
print((care_compare.isnull().sum() / len(care_compare) * 100).round(1).to_string())

print(f"\nData types:")
print(care_compare.dtypes.to_string())

print(f"\nUnique NPIs: {care_compare['NPI'].nunique():,}")

In [ ]:
# Care Compare visualizations

# Telehealth flag
fig = px.bar(
    care_compare["Telehlth"].value_counts().reset_index(),
    x="Telehlth", y="count",
    title="Care Compare: Telehealth Flag Distribution",
)
fig.show()

# Primary specialty distribution
fig = px.bar(
    care_compare["pri_spec"].value_counts().head(20).reset_index(),
    x="pri_spec", y="count",
    title="Care Compare: Top 20 Primary Specialties",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Distinct specialties per NPI (primary + secondary)
spec_cols = ["pri_spec", "sec_spec_1", "sec_spec_2", "sec_spec_3", "sec_spec_4"]
spec_count = care_compare[spec_cols].notna().sum(axis=1)
fig = px.histogram(
    spec_count,
    title="Care Compare: Number of Specialties per Provider",
    labels={"value": "Specialty Count", "count": "Providers"},
)
fig.show()

In [ ]:
# --- Part B EDA ---
print("=" * 60)
print("MEDICARE PART B UTILIZATION")
print("=" * 60)

print(f"\nShape: {partb.shape}")
print(f"\nColumns: {list(partb.columns)}")

print(f"\nNull rates:")
print((partb.isnull().sum() / len(partb) * 100).round(1).to_string())

print(f"\nData types:")
print(partb.dtypes.to_string())

print(f"\nUnique NPIs: {partb['Rndrng_NPI'].nunique():,}")

# Quick stats on numeric columns
for col in ["Tot_Benes", "Tot_Srvcs", "Avg_Sbmtd_Chrg"]:
    vals = pd.to_numeric(partb[col], errors="coerce")
    print(f"\n{col} stats:")
    print(vals.describe().to_string())

In [ ]:
# Part B visualizations

# Top specialties by claim lines
fig = px.bar(
    partb["Rndrng_Prvdr_Type"].value_counts().head(15).reset_index(),
    x="Rndrng_Prvdr_Type", y="count",
    title="Part B: Top 15 Specialties by Claim Lines",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Total charges per NPI (computed as Avg_Sbmtd_Chrg * Tot_Srvcs)
partb["_est_total_chrg"] = (
    pd.to_numeric(partb["Avg_Sbmtd_Chrg"], errors="coerce")
    * pd.to_numeric(partb["Tot_Srvcs"], errors="coerce")
)
npi_charges = partb.groupby("Rndrng_NPI")["_est_total_chrg"].sum().reset_index(name="est_total_charges")

fig = px.histogram(
    npi_charges["est_total_charges"].clip(upper=npi_charges["est_total_charges"].quantile(0.99)),
    nbins=50,
    title="Part B: Estimated Total Charges per NPI (clipped at 99th pctile)",
    labels={"value": "Estimated Total Charges ($)", "count": "Providers"},
)
fig.show()

print(f"Charge stats per NPI:")
print(npi_charges["est_total_charges"].describe().to_string())

# Clean up temp column
partb.drop(columns=["_est_total_chrg"], inplace=True)

In [ ]:
# --- NPPES (MA) EDA ---
print("=" * 60)
print("NPPES — MASSACHUSETTS PROVIDERS")
print("=" * 60)

print(f"\nShape: {nppes_ma.shape}")
print(f"\nColumns: {list(nppes_ma.columns)}")

print(f"\nNull rates:")
print((nppes_ma.isnull().sum() / len(nppes_ma) * 100).round(1).to_string())

print(f"\nEntity Type Code distribution:")
print(nppes_ma["Entity Type Code"].value_counts().to_string())

print(f"\nUnique NPIs: {nppes_ma['NPI'].nunique():,}")

# Check for deactivated NPIs
deactivated = nppes_ma["NPI Deactivation Date"].notna().sum()
print(f"\nDeactivated NPIs: {deactivated:,} ({deactivated/len(nppes_ma)*100:.1f}%)")

In [ ]:
# NPPES MA visualizations

# Provider type (Entity Type 1=Individual, 2=Organization)
fig = px.bar(
    nppes_ma["Entity Type Code"].value_counts().reset_index(),
    x="Entity Type Code", y="count",
    title="NPPES MA: Entity Type (1=Individual, 2=Organization)",
)
fig.show()

# Top taxonomy codes (proxy for specialty)
fig = px.bar(
    nppes_ma["Healthcare Provider Taxonomy Code_1"].value_counts().head(20).reset_index(),
    x="Healthcare Provider Taxonomy Code_1", y="count",
    title="NPPES MA: Top 20 Taxonomy Codes",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# City distribution
fig = px.bar(
    nppes_ma["Provider Business Practice Location Address City Name"].value_counts().head(20).reset_index(),
    x="Provider Business Practice Location Address City Name", y="count",
    title="NPPES MA: Top 20 Cities",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# ZIP code distribution
zip_counts = nppes_ma["Provider Business Practice Location Address Postal Code"].str[:5].value_counts().head(25)
fig = px.bar(
    zip_counts.reset_index(),
    x="Provider Business Practice Location Address Postal Code", y="count",
    title="NPPES MA: Top 25 ZIP Codes",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# --- Part D Prescriber EDA ---
print("=" * 60)
print("MEDICARE PART D PRESCRIBER")
print("=" * 60)

print(f"\nShape: {partd.shape}")
print(f"\nNull rates:")
print((partd.isnull().sum() / len(partd) * 100).round(1).to_string())

print(f"\nUnique NPIs: {partd['PRSCRBR_NPI'].nunique():,}")

for col in ["Tot_Clms", "Tot_Drug_Cst", "Tot_Benes", "Opioid_Prscrbr_Rate"]:
    vals = pd.to_numeric(partd[col], errors="coerce")
    print(f"\n{col}:")
    print(vals.describe().to_string())

In [ ]:
# Part D visualizations

# Top prescriber types
fig = px.bar(
    partd["Prscrbr_Type"].value_counts().head(15).reset_index(),
    x="Prscrbr_Type", y="count",
    title="Part D: Top 15 Prescriber Types",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Total drug cost per NPI distribution
drug_cost = pd.to_numeric(partd["Tot_Drug_Cst"], errors="coerce")
fig = px.histogram(
    drug_cost.clip(upper=drug_cost.quantile(0.99)),
    nbins=50,
    title="Part D: Total Drug Cost per NPI (clipped at 99th pctile)",
)
fig.show()

# Opioid prescribing rate distribution
opioid_rate = pd.to_numeric(partd["Opioid_Prscrbr_Rate"], errors="coerce").dropna()
if len(opioid_rate) > 0:
    fig = px.histogram(opioid_rate, nbins=50, title="Part D: Opioid Prescribing Rate Distribution")
    fig.show()

In [ ]:
# --- Facility Affiliations EDA ---
print("=" * 60)
print("FACILITY AFFILIATIONS (NPI -> CCN CROSSWALK)")
print("=" * 60)

print(f"\nShape: {facility_affil.shape}")
print(f"\nAffiliations per NPI:")
affil_per_npi = facility_affil.groupby("NPI").size()
print(affil_per_npi.describe().to_string())

print(f"\nNPIs with 1 affiliation: {(affil_per_npi == 1).sum():,}")
print(f"NPIs with 2+ affiliations: {(affil_per_npi >= 2).sum():,}")
print(f"NPIs with 5+ affiliations: {(affil_per_npi >= 5).sum():,}")

In [ ]:
# --- Hospital Quality Summary EDA ---
print("=" * 60)
print("HOSPITAL QUALITY SUMMARY")
print("=" * 60)

print(f"\nShape: {hosp_summary.shape}")
print(f"\nNull rates:")
print((hosp_summary.isnull().sum() / len(hosp_summary) * 100).round(1).to_string())

# Star rating distribution
fig = px.bar(
    hosp_summary["hospital_star_rating"].value_counts().sort_index().reset_index(),
    x="hospital_star_rating", y="count",
    title="Hospital Overall Star Rating Distribution",
)
fig.show()

# HVBP TPS distribution
hvbp_vals = pd.to_numeric(hosp_summary["hvbp_total_score"], errors="coerce").dropna()
if len(hvbp_vals) > 0:
    fig = px.histogram(hvbp_vals, nbins=30, title="HVBP Total Performance Score Distribution")
    fig.show()

# Worse-than-national counts
for col in ["infection_measures_worse", "complication_measures_worse", "readmission_measures_worse"]:
    vals = hosp_summary[col]
    print(f"\n{col}:")
    print(vals.describe().to_string())

In [ ]:
# --- Open Payments EDA ---
print("=" * 60)
print("OPEN PAYMENTS (GENERAL PAYMENTS)")
print("=" * 60)

print(f"\nShape: {open_payments.shape}")
print(f"\nPayment amount stats:")
print(open_payments["total_payment_amount"].describe().to_string())

print(f"\nPayment count stats:")
print(open_payments["payment_count"].describe().to_string())

print(f"\nFood & beverage: {open_payments['food_bev_count'].sum():,.0f} payments across {(open_payments['food_bev_count'] > 0).sum():,} NPIs")
print(f"Consulting: {open_payments['consulting_count'].sum():,.0f} payments across {(open_payments['consulting_count'] > 0).sum():,} NPIs")

In [ ]:
# Open Payments visualizations

# Payment amount distribution (clipped)
fig = px.histogram(
    open_payments["total_payment_amount"].clip(upper=open_payments["total_payment_amount"].quantile(0.99)),
    nbins=50,
    title="Open Payments: Total Payment per NPI (clipped at 99th pctile)",
)
fig.show()

# Payment count distribution
fig = px.histogram(
    open_payments["payment_count"].clip(upper=open_payments["payment_count"].quantile(0.99)),
    nbins=50,
    title="Open Payments: Number of Payments per NPI (clipped at 99th pctile)",
)
fig.show()

## Step 3 — Data Preprocessing & Cleaning

**What this does:** Harmonize NPIs, filter to MA, aggregate Part B to per-NPI, deduplicate, and build the master table via left joins.

In [ ]:
# --- Step 3: Data Preprocessing & Cleaning ---

# 1. NPI harmonization across all datasets
def clean_npi(df, npi_col="NPI"):
    """Strip whitespace, cast to string, validate 10-digit format."""
    df[npi_col] = df[npi_col].astype(str).str.strip()
    valid_mask = df[npi_col].str.match(r"^\d{10}$")
    invalid_count = (~valid_mask).sum()
    if invalid_count > 0:
        print(f"  ⚠️ {invalid_count:,} invalid NPIs removed (not 10 digits)")
        df = df[valid_mask].copy()
    return df

print("NPI harmonization:")

print(f"\nNPPES MA: {len(nppes_ma):,} rows before")
nppes_ma = clean_npi(nppes_ma)
print(f"  → {len(nppes_ma):,} rows after")

print(f"\nQPP: {len(qpp):,} rows before")
qpp = clean_npi(qpp, npi_col="NPI")
print(f"  → {len(qpp):,} rows after")

print(f"\nCare Compare: {len(care_compare):,} rows before")
care_compare = clean_npi(care_compare, npi_col="NPI")
print(f"  → {len(care_compare):,} rows after")

print(f"\nPart B: {len(partb):,} rows before")
partb = clean_npi(partb, npi_col="Rndrng_NPI")
print(f"  → {len(partb):,} rows after")

In [ ]:
# 3. QPP cleaning — parse scores and deduplicate

# Parse MIPS final score to numeric
qpp["mips_final_score"] = pd.to_numeric(qpp["final_MIPS_score"], errors="coerce")

# Parse category scores
qpp["quality_score"] = pd.to_numeric(qpp["Quality_category_score"], errors="coerce")
qpp["cost_score"] = pd.to_numeric(qpp["Cost_category_score"], errors="coerce")
qpp["improvement_activities_score"] = pd.to_numeric(qpp["IA_category_score"], errors="coerce")
qpp["promoting_interoperability_score"] = pd.to_numeric(qpp["PI_category_score"], errors="coerce")

# No low-volume flag in QPP data — will infer from Part B
qpp["low_volume_flag"] = None

print(f"MIPS final score stats:")
print(qpp["mips_final_score"].describe().to_string())

# Deduplication — keep highest MIPS score per NPI
# Trade-off: biases upward. A provider under two TINs may have different scores
# reflecting different practice contexts. For MVP we accept this and document it.
dupes = qpp["NPI"].duplicated().sum()
print(f"\nDuplicate NPIs in QPP: {dupes:,}")
if dupes > 0:
    qpp = qpp.sort_values("mips_final_score", ascending=False).drop_duplicates(subset="NPI", keep="first")
    print(f"After dedup (kept highest score): {len(qpp):,} rows")

In [ ]:
# 4. Part B aggregation — per-NPI summary
# Note: Avg_Sbmtd_Chrg is per-service-line average, not total
# Compute estimated totals: avg_charge * num_services

partb["_tot_srvcs"] = pd.to_numeric(partb["Tot_Srvcs"], errors="coerce")
partb["_tot_benes"] = pd.to_numeric(partb["Tot_Benes"], errors="coerce")
partb["_avg_sbmtd_chrg"] = pd.to_numeric(partb["Avg_Sbmtd_Chrg"], errors="coerce")
partb["_avg_mdcr_alowd"] = pd.to_numeric(partb["Avg_Mdcr_Alowd_Amt"], errors="coerce")
partb["_est_total_chrg"] = partb["_avg_sbmtd_chrg"] * partb["_tot_srvcs"]
partb["_est_total_alowd"] = partb["_avg_mdcr_alowd"] * partb["_tot_srvcs"]

partb_agg = partb.groupby("Rndrng_NPI").agg(
    total_services=("_tot_srvcs", "sum"),
    total_beneficiaries=("_tot_benes", "max"),  # max not sum: Tot_Benes is per-service-line, summing double-counts
    est_total_submitted_charges=("_est_total_chrg", "sum"),
    est_total_medicare_allowed=("_est_total_alowd", "sum"),
    distinct_hcpcs=("HCPCS_Cd", "nunique"),
    primary_specialty=("Rndrng_Prvdr_Type", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None),
).reset_index()

partb_agg = partb_agg.rename(columns={"Rndrng_NPI": "NPI"})

print(f"Part B aggregated: {len(partb_agg):,} unique NPIs")
print(f"\nPer-NPI stats:")
print(partb_agg[["total_services", "total_beneficiaries", "est_total_submitted_charges"]].describe().to_string())

In [ ]:
# Part D cleaning — filter to individuals and harmonize NPI
print(f"Part D before filtering: {len(partd):,}")

# Filter to individual providers only (exclude organizations)
partd = partd[partd["Prscrbr_Ent_Cd"] == "I"].copy()
print(f"After filtering to individuals (Ent_Cd=I): {len(partd):,}")

# NPI harmonization
partd = clean_npi(partd, npi_col="PRSCRBR_NPI")

# Convert key columns to numeric
for col in ["Tot_Clms", "Tot_30day_Fills", "Tot_Drug_Cst", "Tot_Day_Suply", "Tot_Benes",
            "Brnd_Tot_Clms", "Brnd_Tot_Drug_Cst", "Gnrc_Tot_Clms", "Gnrc_Tot_Drug_Cst",
            "Opioid_Tot_Clms", "Opioid_Prscrbr_Rate", "Opioid_LA_Prscrbr_Rate",
            "Antbtc_Tot_Clms", "Bene_Avg_Age", "Bene_Avg_Risk_Scre"]:
    partd[col] = pd.to_numeric(partd[col], errors="coerce")

# Check for duplicate NPIs
partd_dupes = partd["PRSCRBR_NPI"].duplicated().sum()
print(f"Duplicate NPIs in Part D (individuals): {partd_dupes:,}")
if partd_dupes > 0:
    partd = partd.drop_duplicates(subset="PRSCRBR_NPI", keep="first")
    print(f"After dedup: {len(partd):,}")

# Compute derived columns
partd["brand_pct"] = (partd["Brnd_Tot_Clms"] / partd["Tot_Clms"] * 100).round(1)
partd["opioid_pct"] = (partd["Opioid_Tot_Clms"] / partd["Tot_Clms"] * 100).round(1)

# Select columns for master join
partd_for_join = partd[["PRSCRBR_NPI", "Tot_Clms", "Tot_Drug_Cst", "Tot_Benes",
                         "brand_pct", "opioid_pct", "Opioid_Prscrbr_Rate",
                         "Bene_Avg_Age", "Bene_Avg_Risk_Scre"]].copy()
partd_for_join = partd_for_join.rename(columns={
    "PRSCRBR_NPI": "NPI",
    "Tot_Clms": "partd_total_claims",
    "Tot_Drug_Cst": "partd_total_drug_cost",
    "Tot_Benes": "partd_total_beneficiaries",
    "brand_pct": "partd_brand_pct",
    "opioid_pct": "partd_opioid_pct",
    "Opioid_Prscrbr_Rate": "partd_opioid_rate",
    "Bene_Avg_Age": "partd_bene_avg_age",
    "Bene_Avg_Risk_Scre": "partd_bene_avg_risk",
})

print(f"Part D ready for join: {len(partd_for_join):,} NPIs")

In [ ]:
# Hospital Quality attribution via Facility Affiliations (NPI -> CCN -> Hospital Quality)

# Clean facility affiliations NPI
facility_affil = clean_npi(facility_affil, npi_col="NPI")
facility_affil = facility_affil.rename(columns={
    "Facility Affiliations Certification Number": "CCN"
})

# Filter to hospital affiliations only
hosp_affil = facility_affil[facility_affil["facility_type"] == "Hospital"][["NPI", "CCN"]].copy()
print(f"Hospital affiliations: {len(hosp_affil):,} (NPI-CCN pairs)")
print(f"Unique NPIs with hospital affiliation: {hosp_affil['NPI'].nunique():,}")
print(f"Unique hospitals (CCNs): {hosp_affil['CCN'].nunique():,}")

# Join facility affiliations to hospital summary on CCN = Facility ID
hosp_for_provider = hosp_affil.merge(
    hosp_summary[["Facility ID", "hospital_star_rating", "hvbp_total_score",
                   "hcahps_star_rating", "infection_measures_worse",
                   "complication_measures_worse", "readmission_measures_worse"]],
    left_on="CCN",
    right_on="Facility ID",
    how="inner"
).drop(columns=["Facility ID"])

matched_ccns = hosp_for_provider["CCN"].nunique()
print(f"CCNs matched to hospital quality data: {matched_ccns:,}")
print(f"NPI-hospital pairs with quality data: {len(hosp_for_provider):,}")

# A provider may be affiliated with multiple hospitals — keep the one with the best star rating
hosp_for_provider["_star_numeric"] = pd.to_numeric(hosp_for_provider["hospital_star_rating"], errors="coerce")
hosp_for_provider = hosp_for_provider.sort_values("_star_numeric", ascending=False).drop_duplicates(
    subset="NPI", keep="first"
).drop(columns=["_star_numeric"])

# Rename for master join
hosp_for_join = hosp_for_provider.rename(columns={
    "CCN": "affiliated_hospital_ccn",
    "hospital_star_rating": "affiliated_hospital_star_rating",
    "hvbp_total_score": "affiliated_hospital_hvbp_score",
    "hcahps_star_rating": "affiliated_hospital_hcahps_rating",
    "infection_measures_worse": "affiliated_hospital_infection_flags",
    "complication_measures_worse": "affiliated_hospital_complication_flags",
    "readmission_measures_worse": "affiliated_hospital_readmission_flags",
})

print(f"\nHospital attribution ready for join: {len(hosp_for_join):,} unique NPIs")

In [ ]:
# Open Payments cleaning — NPI harmonization
open_payments["NPI"] = open_payments["NPI"].astype(str).str.strip()
valid_mask = open_payments["NPI"].str.match(r"^\d{10}$")
invalid = (~valid_mask).sum()
if invalid > 0:
    print(f"Open Payments: {invalid:,} invalid NPIs removed")
    open_payments = open_payments[valid_mask].copy()

# Rename for master join
op_for_join = open_payments.rename(columns={
    "total_payment_amount": "open_payments_total",
    "payment_count": "open_payments_count",
    "food_bev_count": "open_payments_food_bev_count",
    "consulting_count": "open_payments_consulting_count",
})

print(f"Open Payments ready for join: {len(op_for_join):,} NPIs")

In [ ]:
# 5-6. Care Compare dedup and 7. Build master table

# Care Compare deduplication
cc_dupes = care_compare["NPI"].duplicated().sum()
print(f"Duplicate NPIs in Care Compare: {cc_dupes:,}")

if cc_dupes > 0:
    # Keep first occurrence for most columns
    # Aggregate facility-related columns as semicolon-separated lists
    affil_cols = ["Facility Name", "org_pac_id"]
    agg_dict = {}
    for col in care_compare.columns:
        if col == "NPI":
            continue
        elif col in affil_cols:
            agg_dict[col] = lambda x: "; ".join(x.dropna().unique())
        else:
            agg_dict[col] = "first"
    care_compare_dedup = care_compare.groupby("NPI").agg(agg_dict).reset_index()
    print(f"After dedup (affiliations aggregated): {len(care_compare_dedup):,} rows")
else:
    care_compare_dedup = care_compare

# Select Care Compare columns for the master table
# These are enrichment for deferred Step 4 business logic
cc_join_cols = ["NPI", "pri_spec", "Telehlth", "Facility Name", "org_pac_id",
                "num_org_mem", "Cred", "Med_sch", "Grd_yr"]
cc_for_join = care_compare_dedup[[c for c in cc_join_cols if c in care_compare_dedup.columns]]

# Build master table — NPPES MA as left side
master = nppes_ma[["NPI", "Entity Type Code", "Provider Last Name (Legal Name)",
                    "Provider First Name", "Provider Credential Text",
                    "Provider Business Practice Location Address City Name",
                    "Provider Business Practice Location Address Postal Code",
                    "Healthcare Provider Taxonomy Code_1"]].copy()

print(f"\nMaster table start: {len(master):,} MA NPIs")

# Left join QPP
qpp_join_cols = ["NPI", "mips_final_score", "low_volume_flag",
                 "quality_score", "cost_score", "improvement_activities_score",
                 "promoting_interoperability_score", "source"]
master = master.merge(qpp[qpp_join_cols], on="NPI", how="left")
qpp_matched = master["mips_final_score"].notna().sum()
print(f"QPP matched: {qpp_matched:,} / {len(master):,} ({qpp_matched/len(master)*100:.1f}%)")

# Left join Part B
master = master.merge(partb_agg, on="NPI", how="left")
partb_matched = master["total_services"].notna().sum()
print(f"Part B matched: {partb_matched:,} / {len(master):,} ({partb_matched/len(master)*100:.1f}%)")

# Left join Care Compare
master = master.merge(cc_for_join, on="NPI", how="left")
cc_matched = master["pri_spec"].notna().sum()
print(f"Care Compare matched: {cc_matched:,} / {len(master):,} ({cc_matched/len(master)*100:.1f}%)")

# Left join Part D
master = master.merge(partd_for_join, on="NPI", how="left")
partd_matched = master["partd_total_claims"].notna().sum()
print(f"Part D matched: {partd_matched:,} / {len(master):,} ({partd_matched/len(master)*100:.1f}%)")

# Left join Hospital Quality (via facility affiliation CCN crosswalk)
master = master.merge(hosp_for_join, on="NPI", how="left")
hosp_matched = master["affiliated_hospital_star_rating"].notna().sum()
print(f"Hospital Quality matched: {hosp_matched:,} / {len(master):,} ({hosp_matched/len(master)*100:.1f}%)")

# Left join Open Payments
master = master.merge(op_for_join, on="NPI", how="left")
op_matched = master["open_payments_total"].notna().sum()
print(f"Open Payments matched: {op_matched:,} / {len(master):,} ({op_matched/len(master)*100:.1f}%)")

# Validate: row count should match NPPES MA
assert len(master) == len(nppes_ma), (
    f"Master table has {len(master):,} rows but NPPES MA has {len(nppes_ma):,}. "
    "Left join should preserve all NPPES rows. Check for duplicate NPIs in join sources."
)
print(f"\n✓ Master table: {len(master):,} rows (matches NPPES MA count)")

# Column summary
print(f"\nColumn list ({len(master.columns)} total):")
print(list(master.columns))

print(f"\nNull rates:")
print((master.isnull().sum() / len(master) * 100).round(1).sort_values(ascending=False).to_string())

## Output Schema

One row per MA NPI. MIPS scores where available, Part B utilization summary, Care Compare enrichment. This table is the input for the business logic brainstorm (Step 4, to be designed).

## Step 4 — Core Transformation: Clean Schema + Confidence Tiers

**What this does:** Rename columns to the eng handoff schema, drop unnecessary columns, compute the data source count and confidence tier for each NPI. The output is one row per NPI with 39 standardized columns.

**Confidence tier logic:**
- **Tier 1 (government_assessed):** Has MIPS score — CMS directly evaluated this provider
- **Tier 2 (strong_profile):** No MIPS, but 3+ other data sources
- **Tier 3 (partial_profile):** No MIPS, 1-2 other data sources
- **Tier 4 (no_government_data):** No data from any source

In [ ]:
# --- Step 4: Core Transformation ---

# Build final output with clean column names
output = pd.DataFrame()

# Identity
output["npi"] = master["NPI"]
output["entity_type"] = master["Entity Type Code"]
output["provider_name"] = (
    master["Provider Last Name (Legal Name)"].fillna("") + ", " +
    master["Provider First Name"].fillna("")
).str.strip(", ")
output["provider_state"] = "MA"  # All providers in this notebook are MA-filtered from NPPES
output["provider_zip"] = master["Provider Business Practice Location Address Postal Code"]
output["taxonomy_code"] = master["Healthcare Provider Taxonomy Code_1"]

# Government Quality Assessment
output["mips_final_score"] = master["mips_final_score"]
output["mips_quality_score"] = master["quality_score"]
output["mips_cost_score"] = master["cost_score"]
output["mips_ia_score"] = master["improvement_activities_score"]
output["mips_pi_score"] = master["promoting_interoperability_score"]

# Medicare Part B Activity
output["partb_total_services"] = master["total_services"]
output["partb_total_beneficiaries"] = master["total_beneficiaries"]
output["partb_est_total_charges"] = master["est_total_submitted_charges"]
output["partb_distinct_hcpcs"] = master["distinct_hcpcs"]
output["partb_primary_specialty"] = master["primary_specialty"]

# Medicare Part D Prescribing
output["partd_total_claims"] = master["partd_total_claims"]
output["partd_total_drug_cost"] = master["partd_total_drug_cost"]
output["partd_total_beneficiaries"] = master["partd_total_beneficiaries"]
output["partd_opioid_rate"] = master["partd_opioid_rate"]
output["partd_bene_avg_risk"] = master["partd_bene_avg_risk"]

# Provider Context
output["care_compare_specialty"] = master["pri_spec"]
output["care_compare_credential"] = master["Cred"]
output["care_compare_med_school"] = master["Med_sch"]
output["care_compare_grad_year"] = master["Grd_yr"]
output["care_compare_telehealth_flag"] = master["Telehlth"].map({"Y": True, "N": False})
output["care_compare_group_size"] = pd.to_numeric(master["num_org_mem"], errors="coerce")

# Hospital Affiliation Quality
output["affiliated_hospital_ccn"] = master["affiliated_hospital_ccn"]
output["affiliated_hospital_star_rating"] = master["affiliated_hospital_star_rating"]
output["affiliated_hospital_hvbp_score"] = pd.to_numeric(master["affiliated_hospital_hvbp_score"], errors="coerce")
output["affiliated_hospital_hcahps_rating"] = master["affiliated_hospital_hcahps_rating"]
output["affiliated_hospital_infection_flags"] = master["affiliated_hospital_infection_flags"].fillna(0).astype(int)
output["affiliated_hospital_complication_flags"] = master["affiliated_hospital_complication_flags"].fillna(0).astype(int)
output["affiliated_hospital_readmission_flags"] = master["affiliated_hospital_readmission_flags"].fillna(0).astype(int)

# Industry Relationships
output["open_payments_total"] = master["open_payments_total"]
output["open_payments_count"] = master["open_payments_count"]
output["open_payments_consulting_count"] = master["open_payments_consulting_count"]

print(f"Output table: {output.shape[0]:,} rows, {output.shape[1]} columns")
print(f"Columns: {list(output.columns)}")

In [ ]:
# Compute confidence indicators

# Data source count: how many of the 6 sources have data for this NPI
source_indicators = {
    "MIPS": output["mips_final_score"].notna(),
    "Part B": output["partb_total_services"].notna(),
    "Care Compare": output["care_compare_specialty"].notna(),
    "Part D": output["partd_total_claims"].notna(),
    "Hospital Quality": output["affiliated_hospital_star_rating"].notna(),
    "Open Payments": output["open_payments_total"].notna(),
}

output["data_source_count"] = sum(indicator.astype(int) for indicator in source_indicators.values())

# Confidence tier
def assign_tier(row):
    if pd.notna(row["mips_final_score"]):
        return 1  # government_assessed
    elif row["data_source_count"] >= 3:
        return 2  # strong_profile
    elif row["data_source_count"] >= 1:
        return 3  # partial_profile
    else:
        return 4  # no_government_data

output["confidence_tier"] = output.apply(assign_tier, axis=1)

tier_labels = {1: "government_assessed", 2: "strong_profile", 3: "partial_profile", 4: "no_government_data"}
output["confidence_tier_label"] = output["confidence_tier"].map(tier_labels)

# Summary
print(f"Final output: {output.shape[0]:,} rows, {output.shape[1]} columns")
print(f"\nConfidence tier distribution:")
for tier in [1, 2, 3, 4]:
    count = (output["confidence_tier"] == tier).sum()
    print(f"  Tier {tier} ({tier_labels[tier]}): {count:,} ({count/len(output)*100:.1f}%)")

print(f"\nData source count distribution:")
print(output["data_source_count"].value_counts().sort_index().to_string())

# Save final output
output.to_csv("2_datasets/step2_output_final.csv", index=False)
output.to_parquet("2_datasets/step2_output_final.parquet", index=False)
print(f"\nFinal output saved to 2_datasets/step2_output_final.csv and .parquet")
print(f"Total columns: {output.shape[1]}")

In [ ]:
# --- Output: Master table summary (with all data sources) ---
print("=" * 60)
print("MASTER TABLE — READY FOR BUSINESS LOGIC BRAINSTORM")
print("=" * 60)

print(f"\nShape: {master.shape}")

has_mips = master["mips_final_score"].notna().sum()
has_partb = master["total_services"].notna().sum()
has_cc = master["pri_spec"].notna().sum()
has_partd = master["partd_total_claims"].notna().sum()
has_hosp = master["affiliated_hospital_star_rating"].notna().sum()
has_op = master["open_payments_total"].notna().sum()

print(f"\nData source coverage:")
print(f"  MIPS score:        {has_mips:>8,} ({has_mips/len(master)*100:>5.1f}%)")
print(f"  Part B:            {has_partb:>8,} ({has_partb/len(master)*100:>5.1f}%)")
print(f"  Care Compare:      {has_cc:>8,} ({has_cc/len(master)*100:>5.1f}%)")
print(f"  Part D:            {has_partd:>8,} ({has_partd/len(master)*100:>5.1f}%)")
print(f"  Hospital Quality:  {has_hosp:>8,} ({has_hosp/len(master)*100:>5.1f}%)")
print(f"  Open Payments:     {has_op:>8,} ({has_op/len(master)*100:>5.1f}%)")

# Coverage: any data at all
has_any = master[
    (master["mips_final_score"].notna()) |
    (master["total_services"].notna()) |
    (master["pri_spec"].notna()) |
    (master["partd_total_claims"].notna()) |
    (master["affiliated_hospital_star_rating"].notna()) |
    (master["open_payments_total"].notna())
]
truly_dark = len(master) - len(has_any)
print(f"\n  NPIs with ANY data: {len(has_any):,} ({len(has_any)/len(master)*100:.1f}%)")
print(f"  Truly dark (no data): {truly_dark:,} ({truly_dark/len(master)*100:.1f}%)")

# MIPS score distribution for scored providers
if has_mips > 0:
    print(f"\n  MIPS score distribution (scored only):")
    print(f"  {master['mips_final_score'].dropna().describe().to_string()}")

# Save updated checkpoint
master.to_csv("2_datasets/step2_master_table_checkpoint.csv", index=False)
master.to_parquet("2_datasets/step2_master_table_checkpoint.parquet", index=False)
print(f"\nCheckpoint saved to 2_datasets/step2_master_table_checkpoint.csv and .parquet")
print(f"Total columns: {master.shape[1]}")

## Composite Layer: How to Use This Output

This notebook produces a **structured provider profile**, not a blended score. The composite scoring layer (which combines all 7 pipeline steps into a final provider quality score) should consume this output as follows.

### Confidence Tiers and Weight Allocation

The `confidence_tier` field tells the composite how much to trust the government quality dimension for each provider:

| Tier | Label | What the composite should do |
|---|---|---|
| **1** | `government_assessed` | Full weight on this dimension. MIPS score (0-100) is a direct government quality assessment. This is the strongest signal in the entire pipeline. |
| **2** | `strong_profile` | No quality score, but rich activity data. The composite should redistribute the quality dimension weight to other steps (utilization, credentials, patient experience). The Part B, Part D, and hospital affiliation data here is **context**, not a score. Use it for cross-referencing, not for scoring. |
| **3** | `partial_profile` | Limited data. Same weight redistribution as Tier 2, but with lower confidence in cross-reference signals. |
| **4** | `no_government_data` | No government data at all. Redistribute the full quality dimension weight. This provider is likely commercially focused. The absence of data is not a negative signal. |

### Field Classification

**Quality signals** (can directly inform a score):
- `mips_final_score` — the only direct quality measure in this step (Tier 1 only)
- `mips_quality_score`, `mips_cost_score`, `mips_ia_score`, `mips_pi_score` — category breakdowns for deeper analysis

**Facility quality signals** (indirect, attributed from hospital affiliation):
- `affiliated_hospital_star_rating` — CMS overall hospital rating (1-5)
- `affiliated_hospital_hvbp_score` — Value-Based Purchasing performance
- `affiliated_hospital_hcahps_rating` — patient experience at the affiliated hospital
- `affiliated_hospital_infection_flags`, `complication_flags`, `readmission_flags` — negative signals (count of "worse than national" measures)

These are **not** the provider's own quality. They describe where the provider practices. A provider at a 5-star hospital is not necessarily a 5-star provider, but it's a relevant signal, especially when combined with other dimensions.

**Activity indicators** (not quality, but useful context):
- `partb_total_services`, `partb_total_beneficiaries`, `partb_est_total_charges` — Medicare billing volume. Use for: low-volume detection, specialty confirmation, practice size estimation
- `partd_total_claims`, `partd_total_drug_cost`, `partd_total_beneficiaries` — prescribing volume. Use for: confirming the provider is clinically active
- `partd_opioid_rate` — CMS-calculated opioid prescribing rate. Use for: flagging outlier prescribing patterns (not a penalty, a flag for investigation)
- `partd_bene_avg_risk` — average patient risk score. Use for: adjusting expectations (a provider with high-risk patients will have different patterns)

**Provider context** (informational, not scored):
- `care_compare_specialty`, `care_compare_credential`, `care_compare_med_school`, `care_compare_grad_year` — provider background. Feeds into Step 4 (Credentials) more than Step 2
- `care_compare_telehealth_flag` — whether provider offers telehealth
- `care_compare_group_size` — practice size (note: this is the billing group, not necessarily the independent practice size)

**Industry relationships** (transparency signal):
- `open_payments_total`, `open_payments_count` — total industry payments. Not inherently negative, but high concentration is worth flagging
- `open_payments_consulting_count` — consulting fee count. Higher signal than food/beverage payments

### NULL Handling

Most fields will be NULL for most providers. This is expected and documented:
- **MIPS fields:** NULL for 97% of providers. This is the structural ceiling of Medicare FFS data.
- **Part B / Part D fields:** NULL means no Medicare billing activity. Not a negative signal for commercially focused providers.
- **Hospital fields:** NULL means no hospital affiliation in the CMS Facility Affiliations file. Common for outpatient-only providers.
- **Open Payments:** NULL means no industry payments reported. This is neutral.

The composite should **never penalize a NULL**. NULL means "no data," not "bad data." Weight redistribution is the correct response.

### Cross-Reference Logic

For Tier 1 providers with low MIPS scores (below 50), the composite should cross-reference:
- **Step 1 (Safety Gate):** Does this provider have disciplinary actions or exclusions? Low MIPS + clean safety gate = likely administrative burden, not bad care.
- **Step 5 (Patient Experience):** What do patients say? Low MIPS + good reviews = reporting issue. Low MIPS + bad reviews = real quality concern.
- **Hospital affiliation:** If affiliated with a high-rated hospital, the low MIPS is more likely a reporting issue.

This cross-reference is the composite layer's job, not Step 2's. Step 2 provides the data; the composite makes the judgment call.

### Structural Ceiling

All data in this step is Medicare fee-for-service. A physician whose patients are commercially insured will appear "thin" or "dark" in this data because of who they treat, not how well they treat them. The composite should treat Tier 4 (no government data) as an information gap, not a quality gap. Other pipeline steps (credentials, patient experience, employer claims) will carry the signal for these providers.